# Klavye tahmini

Kucuk bir metinden sonraki harfi tahmin etmeye calistim. Pytorch.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from pathlib import Path


### Metin


In [ ]:
text=Path('data/notes.txt').read_text().lower()
chars=sorted(set(text))
stoi=dict(zip(chars,range(len(chars))))
itos=dict(zip(range(len(chars)),chars))
print(len(text),len(chars))


### Pencere


In [ ]:
import numpy as np
ids=np.array([stoi[c] for c in text])
win=20
X=np.lib.stride_tricks.sliding_window_view(ids[:-1],win)
y=ids[win:]
X=torch.tensor(X[:800],dtype=torch.long)
y=torch.tensor(y[:800],dtype=torch.long)
X.shape


### Model


In [ ]:
class Keyboard(nn.Module):
    def __init__(self,n,hid=64):
        super().__init__()
        self.emb=nn.Embedding(n,16)
        self.fc1=nn.Linear(16*win,hid)
        self.fc2=nn.Linear(hid,n)
    def forward(self,x):
        x=self.emb(x).reshape(x.size(0),-1)
        x=torch.relu(self.fc1(x))
        return self.fc2(x)
model=Keyboard(len(chars))
crit=nn.CrossEntropyLoss()
opt=optim.Adam(model.parameters())


In [ ]:
ds=torch.utils.data.TensorDataset(X,y)
loader=torch.utils.data.DataLoader(ds,batch_size=32,shuffle=True)
num_epochs=8
model.train()
for epoch in range(num_epochs):
    tot=0
    n=0
    for batch_x,batch_y in loader:
        opt.zero_grad()
        out=model(batch_x)
        loss=crit(out,batch_y)
        loss.backward()
        opt.step()
        tot+=loss.item()
        n+=1
    print('Epoch',epoch+1,'loss',round(tot/n,4))


### Dene


In [ ]:
model.eval()
seed=text[:win]
x=torch.tensor([[stoi[c] for c in seed]],dtype=torch.long)
with torch.no_grad():
    nxt=itos[int(model(x).argmax(1))]
print(seed,'->',nxt)


### Sonuc

Loss dustu ama metin kisa, bazen ayni harfi tekrarliyor.
